# Notebook 04: Spatial Carbon Dispersion Analysis

**Purpose:** Analyze WHERE pollution is accumulating in the simulated city. We construct a 2D spatial grid from vehicle coordinates and compute grid-cell level statistics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="white")

raw_path = Path("../data/raw/simulation_data.csv")
grid_path = Path("../data/processed/pollution_grid.csv")


### Load Data & Construct Spatial Grid

In [ ]:
df = pd.read_csv(raw_path, skiprows=lambda i: i > 0 and i % 10 != 0)

# Define grid bounds
min_x, max_x = df["x"].min(), df["x"].max()
min_y, max_y = df["y"].min(), df["y"].max()

print(f"X bounds: {min_x} to {max_x}")
print(f"Y bounds: {min_y} to {max_y}")

# Define 30x30 spatial grid
grid_size = 30
x_bins = np.linspace(min_x, max_x, grid_size + 1)
y_bins = np.linspace(min_y, max_y, grid_size + 1)

df["grid_x"] = np.digitize(df["x"], x_bins) - 1
df["grid_y"] = np.digitize(df["y"], y_bins) - 1


### Aggregate Grid Cell Statistics

In [ ]:
grid_agg = df.groupby(["grid_x", "grid_y"]).agg(
    vehicle_count=("vehicle_id", "count"),
    average_speed=("speed", "mean"),
    total_co2=("co2", "sum"),
    average_co2=("co2", "mean"),
    average_waiting_time=("waiting_time", "mean")
).reset_index()

# Scale up count and co2 by sampling rate (10x)
grid_agg["vehicle_count"] = grid_agg["vehicle_count"] * 10
grid_agg["total_co2"] = grid_agg["total_co2"] * 10

grid_agg.to_csv(grid_path, index=False)
print(f"Saved pollution grid data to: {grid_path}")


### Plot Carbon Spatial Heatmap

In [ ]:
# Create 2D pivot for plotting
heatmap_data = grid_agg.pivot(index="grid_y", columns="grid_x", values="total_co2").fillna(0)

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, cmap="hot", cbar_kws={'label': 'Total CO2 (mg/s)'})
plt.title("Carbon Dispersion Spatial Heatmap")
plt.xlabel("Grid X cell")
plt.ylabel("Grid Y cell")
plt.gca().invert_yaxis()
plt.savefig("../reports/figures/spatial/carbon_spatial_heatmap.png", dpi=300)
plt.show()


### Identify Top Pollution Hotspots

In [ ]:
top_hotspots = grid_agg.sort_values(by="total_co2", ascending=False).head(5)
print("Top 5 Pollution Grid Cell Hotspots:")
print(top_hotspots)

# Top road segments by CO2
road_agg = df.groupby("road_id")["co2"].sum().reset_index()
road_agg["co2"] = road_agg["co2"] * 10
top_roads = road_agg.sort_values(by="co2", ascending=False).head(5)
print("\nTop 5 High-Emission Road Segments:")
print(top_roads)
